# Train-derived keyword/regex baseline — v2 (no-refusal) dataset

**Ovo je train-derived keyword/regex baseline** — regex obrasci nisu ručno napisani niti preuzeti iz
spoljne liste harmful reči, i ovo NIJE neuronski/ML model. Svi izrazi su automatski izvedeni
isključivo iz `train.jsonl` group-aware statistikom (document frequency po `original_idx` grupi +
smoothed log-odds), po unapred zaključanim pravilima implementiranim ispod.

Cilj: jednostavan, reproduktibilan baseline za v2 harmful/unharmful klasifikaciju (prompt+response),
kao donja granica za poređenje sa Gemma zero-shot/LoRA i Qwen3Guard sistemima. Nema GPU-a, nema
LLM-a, nema prevođenja/stemminga/embeddings-a, nema ručnog dekodiranja obfuskacija, nema spoljnih
rečnika harmful reči — sve što baseline "zna" dolazi isključivo iz `train.jsonl`.


In [1]:
import json, re, unicodedata, math, hashlib, time
from collections import Counter
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 60)

t0 = time.time()

V2_DIR = Path("/home/mls01/data/gemma_v2_no_refusal")
TRAIN_PATH = V2_DIR / "train.jsonl"
VAL_PATH = V2_DIR / "validation.jsonl"
TEST_PATH = V2_DIR / "test.jsonl"  # NOT read until the config-locking cell below
OUT_DIR = Path("/home/mls01/scripts/model/results/regex_baseline_v2_no_refusal")
OUT_DIR.mkdir(parents=True, exist_ok=True)

POSITIVE = "harmful"
NGRAM_SIZES = (1, 2, 3)
MIN_HARMFUL_GROUP_DF = 3
ALPHA = 0.5  # fixed smoothing constant (Jeffreys-style add-0.5), documented in the ranking formula below
TOP_K_GRID = [25, 50, 100, 200, 400]
MIN_DISTINCT_MATCHES_GRID = [1, 2]
print("Constants locked. Output folder:", OUT_DIR)

Constants locked. Output folder: /home/mls01/scripts/model/results/regex_baseline_v2_no_refusal


## 1. Dataset — load + validate train/validation

Test skup se NE učitava u ovoj sekciji (učitava se tek nakon zaključavanja konfiguracije, u sekciji 6).

In [2]:
train_df = pd.read_json(TRAIN_PATH, lines=True)
val_df = pd.read_json(VAL_PATH, lines=True)

assert len(train_df) == 1985 and train_df["original_idx"].nunique() == 800, \
    f"train: očekivano 1985/800, dobijeno {len(train_df)}/{train_df['original_idx'].nunique()}"
assert len(val_df) == 259 and val_df["original_idx"].nunique() == 100, \
    f"validation: očekivano 259/100, dobijeno {len(val_df)}/{val_df['original_idx'].nunique()}"
assert train_df["row_id"].is_unique and val_df["row_id"].is_unique
assert not (set(train_df["row_id"]) & set(val_df["row_id"])), "row_id se preklapa train/validation"
assert not (set(train_df["original_idx"]) & set(val_df["original_idx"])), "original_idx se preklapa train/validation"

allowed = {"harmful", "unharmful"}
assert set(train_df["final_label"].unique()) <= allowed
assert set(val_df["final_label"].unique()) <= allowed
assert (train_df.groupby("original_idx")["final_label"].nunique() == 1).all(), \
    "final_label nije konzistentan unutar neke train original_idx grupe"
assert (val_df.groupby("original_idx")["final_label"].nunique() == 1).all(), \
    "final_label nije konzistentan unutar neke validation original_idx grupe"

print(f"[OK] train: {len(train_df)} redova / {train_df['original_idx'].nunique()} grupa")
print(f"[OK] validation: {len(val_df)} redova / {val_df['original_idx'].nunique()} grupa")
print("[OK] row_id jedinstven u oba splita, original_idx bez preklapanja, "
      "final_label in {'harmful','unharmful'}, final_label konzistentan po original_idx grupi.")

[OK] train: 1985 redova / 800 grupa
[OK] validation: 259 redova / 100 grupa
[OK] row_id jedinstven u oba splita, original_idx bez preklapanja, final_label in {'harmful','unharmful'}, final_label konzistentan po original_idx grupi.


## 2. Zaključana normalizacija

1. Unicode NFKC
2. lowercase
3. zamena interpunkcije/ostalih non-word separatora jednim razmakom
4. spajanje višestrukih razmaka
5. uklanjanje početnih/završnih razmaka

Bez prevođenja, stemming/lemmatizacije, LLM-a, semantic embeddings-a, ručnog dekodiranja obfuskacija
i spoljnih rečnika. `response=None` se tretira kao `""`.

Regex obrasci se konstruišu pomoću `re.escape` uokvirenog `(?<!\w)...(?!\w)` granicama, tako da
kratak izraz ne može pogoditi deo duže reči (npr. `cat` ne pogađa `category`).

In [3]:
def normalize_text(text):
    if text is None:
        text = ""
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower()
    text = re.sub(r"[^\w]+", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def tokenize(norm_text):
    return norm_text.split(" ") if norm_text else []

def ngram_set(tokens, sizes=NGRAM_SIZES):
    s = set()
    L = len(tokens)
    for n in sizes:
        if L < n:
            continue
        for i in range(L - n + 1):
            s.add(" ".join(tokens[i:i + n]))
    return s

def build_pattern(expression):
    return re.compile(r"(?<!\w)" + re.escape(expression) + r"(?!\w)", flags=re.UNICODE)

print(normalize_text("Ⓐⓑⓒ Test!!  multiple   spaces, Punct."), "|", normalize_text(None))

abc test multiple spaces punct | 


## 3. Group-aware kandidat ekstrakcija (samo train)

Za svaki train red: unija word n-grama (1,2,3) iz normalizovanog `prompt` ∪ `response` (svaki izraz
doprinosi najviše jednom po redu). Zatim, po `original_idx` grupi: unija tih skupova preko svih
redova te grupe (svaki izraz doprinosi najviše jednom po grupi, bez obzira na broj redova/augmentacija
ili broj ponavljanja unutar reda). Ovo sprečava da grupe sa više augmentacija imaju veću težinu.

Kandidat mora: `harmful_group_df >= 3`, `harmful_group_prevalence > unharmful_group_prevalence`, i ne
biti potpuno numerički izraz.

In [4]:
train_df["prompt_norm"] = train_df["prompt"].map(normalize_text)
train_df["response_norm"] = train_df["response"].map(normalize_text)
train_df["row_ngrams"] = train_df.apply(
    lambda r: ngram_set(tokenize(r["prompt_norm"])) | ngram_set(tokenize(r["response_norm"])), axis=1)

group_label = train_df.groupby("original_idx")["final_label"].first()
group_ngrams = train_df.groupby("original_idx")["row_ngrams"].apply(lambda sets: set().union(*sets))

harmful_groups = group_label[group_label == POSITIVE].index
unharmful_groups = group_label[group_label != POSITIVE].index
N_HARMFUL_GROUPS = len(harmful_groups)
N_UNHARMFUL_GROUPS = len(unharmful_groups)

harmful_counter = Counter()
for gid in harmful_groups:
    harmful_counter.update(group_ngrams[gid])
unharmful_counter = Counter()
for gid in unharmful_groups:
    unharmful_counter.update(group_ngrams[gid])

all_expr = set(harmful_counter) | set(unharmful_counter)
print(f"n_harmful_train_groups={N_HARMFUL_GROUPS}, n_unharmful_train_groups={N_UNHARMFUL_GROUPS}, "
      f"total distinct n-grams across all train groups={len(all_expr)}. t={time.time()-t0:.1f}s")

n_harmful_train_groups=400, n_unharmful_train_groups=400, total distinct n-grams across all train groups=497428. t=1.0s


## 4. Rangiranje: smoothed log-odds (formula zaključana)

```
smoothed_log_odds = ln((a+ALPHA)/(H-a+ALPHA)) - ln((b+ALPHA)/(U-b+ALPHA))
a = harmful_group_df, b = unharmful_group_df
H = n_harmful_train_groups, U = n_unharmful_train_groups, ALPHA = 0.5 (fiksno, Jeffreys-stil add-0.5)
```

Kandidati su rangirani opadajuće po `smoothed_log_odds`; tie-break: veći `harmful_group_df`, zatim
alfabetski po izrazu (radi potpune determinističnosti — Python `set` iteracija nije garantovano
stabilna između procesa zbog string hash randomizacije). Rangiranje se NE menja ručno nakon
sortiranja.

In [5]:
rows = []
for e in all_expr:
    a = harmful_counter.get(e, 0)
    b = unharmful_counter.get(e, 0)
    if a < MIN_HARMFUL_GROUP_DF:
        continue
    hp = a / N_HARMFUL_GROUPS
    up = b / N_UNHARMFUL_GROUPS
    if not (hp > up):
        continue
    if e.replace(" ", "").isdigit():
        continue
    log_odds = (math.log(a + ALPHA) - math.log(N_HARMFUL_GROUPS - a + ALPHA)
                - math.log(b + ALPHA) + math.log(N_UNHARMFUL_GROUPS - b + ALPHA))
    rows.append({
        "expression": e, "ngram_size": e.count(" ") + 1,
        "harmful_group_df": a, "unharmful_group_df": b,
        "harmful_group_prevalence": hp, "unharmful_group_prevalence": up,
        "smoothed_log_odds": log_odds,
    })

candidate_df = pd.DataFrame(rows).sort_values(
    by=["smoothed_log_odds", "harmful_group_df", "expression"], ascending=[False, False, True]
).reset_index(drop=True)
candidate_df.insert(0, "rank", candidate_df.index + 1)
candidate_df.to_csv(OUT_DIR / "candidate_patterns.csv", index=False)

print(f"[OK] {len(candidate_df)} train candidates after filters "
      f"(min_harmful_group_df>={MIN_HARMFUL_GROUP_DF}, harmful_prev>unharmful_prev, not-fully-numeric). "
      f"t={time.time()-t0:.1f}s")
print("ngram_size distribution among candidates:", candidate_df["ngram_size"].value_counts().to_dict())
candidate_df.head(15)

[OK] 16139 train candidates after filters (min_harmful_group_df>=3, harmful_prev>unharmful_prev, not-fully-numeric). t=1.4s
ngram_size distribution among candidates: {2: 6486, 3: 5438, 1: 4215}


,rank,expression,ngram_size,harmful_group_df,unharmful_group_df,harmful_group_prevalence,unharmful_group_prevalence,smoothed_log_odds
0,1,a t,2,68,0,0.1700,0.0,5.106055
1,2,a n,2,65,0,0.1625,0.0,5.052289
2,3,i n,2,64,0,0.1600,0.0,5.033928
3,4,e n,2,61,0,0.1525,0.0,4.977424
4,5,o n,2,61,0,0.1525,0.0,4.977424
5,6,a l,2,60,0,0.1500,0.0,4.958089
6,7,t o,2,60,0,0.1500,0.0,4.958089
7,8,e r,2,58,0,0.1450,0.0,4.918616
8,9,a r,2,57,0,0.1425,0.0,4.898459
9,10,n t,2,57,0,0.1425,0.0,4.898459


## 5. Dijagnostika: zašto su top kandidati kratki fragmenti (agregirano, bez sirovog teksta)

Pre nego što nastavimo, vredi programski provariti odakle dolazi rang na vrhu liste — ovo je nalaz
o *strukturi dataset-a*, ne o kodu, i objašnjava zašto će zaključana lista izgledati kako izgleda.

In [6]:
obf_mask_train = train_df["augmentation_type"].str.contains("obfuscation", na=False)
print("train: augmentation_type sadrži 'obfuscation' -> final_label distribucija:",
      train_df.loc[obf_mask_train, "final_label"].value_counts().to_dict(),
      f"(n={int(obf_mask_train.sum())})")

candidate_df["_frag_len"] = candidate_df["expression"].str.replace(" ", "", regex=False).str.len()
for k in TOP_K_GRID:
    top = candidate_df.head(k)
    short = int((top["_frag_len"] <= 2).sum())
    print(f"  top_{k}: {short}/{len(top)} ({short/len(top)*100:.0f}%) su fragmenti <=2 karaktera (bez razmaka)")

real_word_ranks = candidate_df.loc[candidate_df["_frag_len"] >= 4, "rank"]
first_real_word_rank = int(real_word_ranks.min()) if len(real_word_ranks) else None
example_real_words = candidate_df[candidate_df["rank"].between(first_real_word_rank, first_real_word_rank + 400)]
example_real_words = example_real_words[example_real_words["_frag_len"] >= 4].head(8)["expression"].tolist()
print(f"\nprvi kandidat sa >=4 karaktera (bez razmaka) je na rangu {first_real_word_rank} "
      f"(van opsega svakog testiranog top_k <= {max(TOP_K_GRID)}). primeri: {example_real_words}")
candidate_df.drop(columns=["_frag_len"], inplace=True)

train: augmentation_type sadrži 'obfuscation' -> final_label distribucija: {'harmful': 405} (n=405)
  top_25: 25/25 (100%) su fragmenti <=2 karaktera (bez razmaka)
  top_50: 49/50 (98%) su fragmenti <=2 karaktera (bez razmaka)
  top_100: 90/100 (90%) su fragmenti <=2 karaktera (bez razmaka)
  top_200: 142/200 (71%) su fragmenti <=2 karaktera (bez razmaka)
  top_400: 221/400 (55%) su fragmenti <=2 karaktera (bez razmaka)

prvi kandidat sa >=4 karaktera (bez razmaka) je na rangu 558 (van opsega svakog testiranog top_k <= 400). primeri: ['targeting', 'request as it', 'strongly', 'derogatory', 'against ethical', 'for educational', 'let us', 'classified']


**Nalaz** (potvrđen programski, ne pretpostavka): u v2 datasetu je `augmentation_type` koji
sadrži `obfuscation` **isključivo** `final_label='harmful'`. Neki obfuskacioni stilovi (npr.
`wide_spacing`, `vaporwave`) ubacuju bukvalne razmake između svakog slova originalnog teksta — pod
propisanom jednostavnom whitespace-tokenizacijom to razbija reči na pojedinačna slova, pa
karakter-bigrami/trigrami (npr. `"a t"`) postaju veštački "harmful-prediktivni" jer se ta obfuskacija
dešava samo na harmful sadržaju, ne zato što nose harmful semantiku. Ovo NIJE bug — to je stvarno
svojstvo dataset-a + propisane (namerno jednostavne, ne-obfuskacija-svesne) normalizacije. Pratimo
ovo dalje u error-analizi (sekcija 8) i u Ograničenjima (REPORT.md).

## 6. Validation grid search (top_k × min_distinct_matches)

Test skup i dalje NIJE učitan.

In [7]:
def compute_metrics(y_true, y_pred, positive=POSITIVE):
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == positive and p == positive)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t != positive and p == positive)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == positive and p != positive)
    tn = sum(1 for t, p in zip(y_true, y_pred) if t != positive and p != positive)
    n = len(y_true)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    fnr = fn / (fn + tp) if (fn + tp) else 0.0
    accuracy = (tp + tn) / n if n else 0.0
    balanced_accuracy = (recall + specificity) / 2
    mcc_denom = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn - fp * fn) / mcc_denom) if mcc_denom else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "accuracy": accuracy, "specificity": specificity, "fpr": fpr, "fnr": fnr,
            "balanced_accuracy": balanced_accuracy, "mcc": mcc,
            "invalid_count": 0, "invalid_rate": 0.0, "total": n}

In [8]:
max_top_k = max(TOP_K_GRID)
n_available = len(candidate_df)
max_patterns_needed = min(max_top_k, n_available)
pattern_pool = candidate_df.head(max_patterns_needed).copy()
pattern_pool["compiled"] = pattern_pool["expression"].map(build_pattern)

val_df["prompt_norm"] = val_df["prompt"].map(normalize_text)
val_df["response_norm"] = val_df["response"].map(normalize_text)

def match_matrix(df, pattern_pool_df):
    hit_prompt, hit_response = {}, {}
    for _, row in pattern_pool_df.iterrows():
        rx = row["compiled"]
        hit_prompt[row["rank"]] = df["prompt_norm"].apply(lambda s: bool(rx.search(s))).values
        hit_response[row["rank"]] = df["response_norm"].apply(lambda s: bool(rx.search(s))).values
    return pd.DataFrame(hit_prompt, index=df.index), pd.DataFrame(hit_response, index=df.index)

t1 = time.time()
val_hit_prompt, val_hit_response = match_matrix(val_df, pattern_pool)
val_hit_any = val_hit_prompt | val_hit_response
print(f"[OK] validation match matrix built {val_hit_prompt.shape} in {time.time()-t1:.1f}s")

search_rows = []
for top_k in TOP_K_GRID:
    if top_k > n_available:
        print(f"  [SKIP] top_k={top_k} > {n_available} raspoloživih kandidata")
        continue
    ranks_subset = list(range(1, top_k + 1))
    distinct_match_count = val_hit_any[ranks_subset].sum(axis=1)
    for min_dm in MIN_DISTINCT_MATCHES_GRID:
        preds = (distinct_match_count >= min_dm).map({True: "harmful", False: "unharmful"})
        m = compute_metrics(val_df["final_label"].tolist(), preds.tolist())
        search_rows.append({"top_k_patterns": top_k, "min_distinct_matches": min_dm,
                            "n_patterns_used": top_k, **m})

validation_search_df = pd.DataFrame(search_rows)
validation_search_df[["top_k_patterns", "min_distinct_matches", "precision", "recall", "f1",
                      "fpr", "n_patterns_used"]]

[OK] validation match matrix built (259, 400) in 1.8s


,top_k_patterns,min_distinct_matches,precision,recall,f1,fpr,n_patterns_used
0,25,1,0.909091,0.12500,0.219780,0.020202,25
1,25,2,1.000000,0.12500,0.222222,0.000000,25
2,50,1,0.833333,0.12500,0.217391,0.040404,50
3,50,2,1.000000,0.12500,0.222222,0.000000,50
4,100,1,0.800000,0.12500,0.216216,0.050505,100
5,100,2,1.000000,0.12500,0.222222,0.000000,100
6,200,1,0.807692,0.13125,0.225806,0.050505,200
7,200,2,0.909091,0.12500,0.219780,0.020202,200
8,400,1,0.812500,0.16250,0.270833,0.060606,400
9,400,2,0.909091,0.12500,0.219780,0.020202,400


## 7. Izbor konfiguracije i zaključavanje (pre učitavanja test skupa)

Redosled izbora: max F1 → max recall → min FPR → min broj obrazaca → veći `min_distinct_matches`
na potpunom tie-u. Nakon ove ćelije se ništa više ne menja: normalizacija, lista obrazaca, threshold
i parser su zaključani.

In [9]:
selection_df = validation_search_df.sort_values(
    by=["f1", "recall", "fpr", "n_patterns_used", "min_distinct_matches"],
    ascending=[False, False, True, True, False],
).reset_index(drop=True)
selected = selection_df.iloc[0]
SELECTED_TOP_K = int(selected["top_k_patterns"])
SELECTED_MIN_DM = int(selected["min_distinct_matches"])
print(f"[SELECTED] top_k_patterns={SELECTED_TOP_K}, min_distinct_matches={SELECTED_MIN_DM} "
      f"(val F1={selected['f1']:.4f}, recall={selected['recall']:.4f}, FPR={selected['fpr']:.4f})")

locked_patterns_df = candidate_df.head(SELECTED_TOP_K).copy().reset_index(drop=True)
locked_expressions_ordered = locked_patterns_df["expression"].tolist()
locked_hash = hashlib.sha256(
    json.dumps(locked_expressions_ordered, ensure_ascii=False).encode("utf-8")
).hexdigest()
print(f"[LOCK] {len(locked_patterns_df)} patterns locked. SHA256={locked_hash}")

locked_val_row = validation_search_df[
    (validation_search_df["top_k_patterns"] == SELECTED_TOP_K) &
    (validation_search_df["min_distinct_matches"] == SELECTED_MIN_DM)
].iloc[0].to_dict()

locked_regex_config = {
    "baseline_name": "Train-derived keyword/regex baseline",
    "normalization": {
        "steps": ["Unicode NFKC", "lowercase",
                  "replace punctuation/non-word separators with single space (regex [^\\w]+ -> ' ', re.UNICODE)",
                  "collapse multiple spaces", "strip leading/trailing spaces"],
        "excludes": ["translation", "stemming/lemmatization", "LLM", "semantic embeddings",
                    "manual obfuscation decoding", "external harmful-word lists"],
    },
    "candidate_extraction": {
        "ngram_sizes": list(NGRAM_SIZES),
        "source": "train.jsonl only (prompt + response), union per row then union per original_idx group",
        "min_harmful_group_df": MIN_HARMFUL_GROUP_DF,
        "prevalence_filter": "harmful_group_prevalence > unharmful_group_prevalence (strict)",
        "numeric_filter": "expression.replace(' ','').isdigit() -> excluded",
    },
    "group_aware_statistics": {
        "n_harmful_train_groups": N_HARMFUL_GROUPS, "n_unharmful_train_groups": N_UNHARMFUL_GROUPS,
        "note": "each expression counted at most once per original_idx group, regardless of how many "
                "rows or how many times it appears within that group",
    },
    "smoothing_formula": {
        "alpha": ALPHA,
        "formula": "smoothed_log_odds = ln((a+alpha)/(H-a+alpha)) - ln((b+alpha)/(U-b+alpha)), "
                  "a=harmful_group_df, b=unharmful_group_df, H=n_harmful_train_groups, U=n_unharmful_train_groups",
    },
    "ngram_range": [min(NGRAM_SIZES), max(NGRAM_SIZES)],
    "top_k_patterns_grid": TOP_K_GRID,
    "min_distinct_matches_grid": MIN_DISTINCT_MATCHES_GRID,
    "selected_top_k_patterns": SELECTED_TOP_K,
    "selected_min_distinct_matches": SELECTED_MIN_DM,
    "selection_order": ["max harmful F1", "max harmful recall", "min FPR", "min pattern count",
                        "max min_distinct_matches (final tie-break)"],
    "validation_metrics_selected_config": {k: locked_val_row[k] for k in
        ["precision", "recall", "f1", "tp", "fp", "fn", "tn", "accuracy", "specificity", "fpr", "fnr",
         "balanced_accuracy", "mcc", "invalid_count", "invalid_rate"]},
    "n_candidates_total": int(len(candidate_df)),
    "locked_patterns_sha256": locked_hash,
}

(OUT_DIR / "locked_regex_config.json").write_text(json.dumps(locked_regex_config, indent=2, ensure_ascii=False))
locked_patterns_df.drop(columns=[c for c in ["compiled"] if c in locked_patterns_df.columns]).to_csv(
    OUT_DIR / "locked_patterns.csv", index=False)
validation_search_df.to_csv(OUT_DIR / "validation_search.csv", index=False)
print("[LOCK] locked_regex_config.json / locked_patterns.csv / validation_search.csv written.")

[SELECTED] top_k_patterns=400, min_distinct_matches=1 (val F1=0.2708, recall=0.1625, FPR=0.0606)
[LOCK] 400 patterns locked. SHA256=add95ede20d89015825c619821df840a12390456bd62896b0bfe80739fc16993
[LOCK] locked_regex_config.json / locked_patterns.csv / validation_search.csv written.


---
## 8. KONFIGURACIJA JE ZAKLJUČANA. Test skup se učitava tek sada.

Od ove tačke nadalje: normalizacija, lista obrazaca, threshold i parser se ne menjaju.

In [10]:
locked_patterns_df["compiled"] = locked_patterns_df["expression"].map(build_pattern)

def run_locked_regex(df):
    df = df.copy()
    df["prompt_norm"] = df["prompt"].map(normalize_text)
    df["response_norm"] = df["response"].map(normalize_text)
    match_counts, matched_patterns_list, matched_in_prompt, matched_in_response = [], [], [], []
    for _, row in df.iterrows():
        hit_exprs, in_prompt, in_response = [], False, False
        for _, pat_row in locked_patterns_df.iterrows():
            rx = pat_row["compiled"]
            p_hit = bool(rx.search(row["prompt_norm"]))
            r_hit = bool(rx.search(row["response_norm"]))
            if p_hit or r_hit:
                hit_exprs.append(pat_row["expression"])
                in_prompt = in_prompt or p_hit
                in_response = in_response or r_hit
        match_counts.append(len(hit_exprs))
        matched_patterns_list.append("; ".join(hit_exprs))
        matched_in_prompt.append(in_prompt)
        matched_in_response.append(in_response)
    df["match_count"] = match_counts
    df["matched_patterns"] = matched_patterns_list
    df["matched_in_prompt"] = matched_in_prompt
    df["matched_in_response"] = matched_in_response
    df["prediction"] = (df["match_count"] >= SELECTED_MIN_DM).map({True: "harmful", False: "unharmful"})
    return df

t2 = time.time()
validation_results_full = run_locked_regex(val_df)
val_m = compute_metrics(validation_results_full["final_label"].tolist(), validation_results_full["prediction"].tolist())
print(f"[OK] validation (locked config) evaluated in {time.time()-t2:.1f}s")
print("validation metrics:", {k: round(v, 4) if isinstance(v, float) else v for k, v in val_m.items()})
assert abs(val_m["f1"] - locked_val_row["f1"]) < 1e-9, "Locked eval must reproduce the selected grid-search row exactly."
print("[OK] Reproducira tačno izabrani grid-search red (sanity check).")

[OK] validation (locked config) evaluated in 5.2s
validation metrics: {'precision': 0.8125, 'recall': 0.1625, 'f1': 0.2708, 'tp': 26, 'fp': 6, 'fn': 134, 'tn': 93, 'accuracy': 0.4595, 'specificity': 0.9394, 'fpr': 0.0606, 'fnr': 0.8375, 'balanced_accuracy': 0.5509, 'mcc': 0.1505, 'invalid_count': 0, 'invalid_rate': 0.0, 'total': 259}
[OK] Reproducira tačno izabrani grid-search red (sanity check).


In [11]:
test_df = pd.read_json(TEST_PATH, lines=True)
assert len(test_df) == 227 and test_df["original_idx"].nunique() == 100, \
    f"test: očekivano 227/100, dobijeno {len(test_df)}/{test_df['original_idx'].nunique()}"
assert test_df["row_id"].is_unique
assert not (set(test_df["row_id"]) & set(train_df["row_id"]))
assert not (set(test_df["row_id"]) & set(val_df["row_id"]))
assert not (set(test_df["original_idx"]) & set(train_df["original_idx"]))
assert not (set(test_df["original_idx"]) & set(val_df["original_idx"]))
assert set(test_df["final_label"].unique()) <= allowed
assert (test_df.groupby("original_idx")["final_label"].nunique() == 1).all()
print(f"[OK] test: {len(test_df)} redova / {test_df['original_idx'].nunique()} grupa -- "
      f"provere prošle, bez preklapanja sa train/validation.")

hash_before_test = hashlib.sha256(
    json.dumps(locked_patterns_df["expression"].tolist(), ensure_ascii=False).encode("utf-8")
).hexdigest()
assert hash_before_test == locked_hash

t3 = time.time()
test_results_full = run_locked_regex(test_df)
print(f"[OK] test (locked config) evaluated ONE TIME in {time.time()-t3:.1f}s")

hash_after_test = hashlib.sha256(
    json.dumps(locked_patterns_df["expression"].tolist(), ensure_ascii=False).encode("utf-8")
).hexdigest()
assert hash_after_test == locked_hash, "Locked pattern list changed during test evaluation!"
print(f"[OK] Hash zaključane liste nepromenjen pre/posle testa: {hash_after_test == locked_hash} ({hash_after_test})")

test_m = compute_metrics(test_results_full["final_label"].tolist(), test_results_full["prediction"].tolist())
print("TEST metrics (evaluated once):", {k: round(v, 4) if isinstance(v, float) else v for k, v in test_m.items()})

[OK] test: 227 redova / 100 grupa -- provere prošle, bez preklapanja sa train/validation.


[OK] test (locked config) evaluated ONE TIME in 5.0s
[OK] Hash zaključane liste nepromenjen pre/posle testa: True (add95ede20d89015825c619821df840a12390456bd62896b0bfe80739fc16993)
TEST metrics (evaluated once): {'precision': 0.3333, 'recall': 0.0315, 'f1': 0.0576, 'tp': 4, 'fp': 8, 'fn': 123, 'tn': 92, 'accuracy': 0.4229, 'specificity': 0.92, 'fpr': 0.08, 'fnr': 0.9685, 'balanced_accuracy': 0.4757, 'mcc': -0.1076, 'invalid_count': 0, 'invalid_rate': 0.0, 'total': 227}


## 9. Confusion matrice, FP/FN, i čuvanje rezultata

Regex baseline nema invalid izlaz (uvek `harmful` ili `unharmful`), pa su valid-only i end-to-end
metrike identične (`invalid_count=0`, `invalid_rate=0`).

In [12]:
ERROR_COLS = ["row_id", "original_idx", "prompt", "response", "final_label", "prediction",
              "match_count", "matched_patterns", "matched_in_prompt", "matched_in_response",
              "language", "augmentation_type", "adversarial"]

def confusion_and_errors(results_full, tag):
    m = compute_metrics(results_full["final_label"].tolist(), results_full["prediction"].tolist())
    cm = pd.DataFrame([[m["tp"], m["fn"]], [m["fp"], m["tn"]]],
                      index=["true_harmful", "true_unharmful"], columns=["pred_harmful", "pred_unharmful"])
    fp_mask = (results_full["final_label"] != POSITIVE) & (results_full["prediction"] == POSITIVE)
    fn_mask = (results_full["final_label"] == POSITIVE) & (results_full["prediction"] != POSITIVE)
    fps = results_full.loc[fp_mask, ERROR_COLS].reset_index(drop=True)
    fns = results_full.loc[fn_mask, ERROR_COLS].reset_index(drop=True)
    print(f"[{tag}] TP={m['tp']} FP={m['fp']} FN={m['fn']} TN={m['tn']} (FP rows={len(fps)}, FN rows={len(fns)})")
    return m, cm, fps, fns

val_m, val_cm, val_fp, val_fn = confusion_and_errors(validation_results_full, "validation")
test_m, test_cm, test_fp, test_fn = confusion_and_errors(test_results_full, "test")

(OUT_DIR / "validation_metrics.json").write_text(json.dumps(val_m, indent=2))
(OUT_DIR / "test_metrics.json").write_text(json.dumps(test_m, indent=2))
val_cm.to_csv(OUT_DIR / "validation_confusion_matrix.csv")
test_cm.to_csv(OUT_DIR / "test_confusion_matrix.csv")
val_fp.to_csv(OUT_DIR / "validation_false_positives.csv", index=False)
val_fn.to_csv(OUT_DIR / "validation_false_negatives.csv", index=False)
test_fp.to_csv(OUT_DIR / "test_false_positives.csv", index=False)
test_fn.to_csv(OUT_DIR / "test_false_negatives.csv", index=False)

SAVE_COLS = ["row_id", "original_idx", "final_label", "prediction", "match_count", "matched_patterns",
            "matched_in_prompt", "matched_in_response", "language", "augmentation_type", "adversarial"]
validation_results_full[SAVE_COLS].to_csv(OUT_DIR / "validation_results_full.csv", index=False)
test_results_full[SAVE_COLS].to_csv(OUT_DIR / "test_results_full.csv", index=False)
print("[OK] validation/test results, metrics, confusion matrices, FP/FN CSVs written.")
print("\ntest confusion matrix:")
test_cm

[validation] TP=26 FP=6 FN=134 TN=93 (FP rows=6, FN rows=134)
[test] TP=4 FP=8 FN=123 TN=92 (FP rows=8, FN rows=123)
[OK] validation/test results, metrics, confusion matrices, FP/FN CSVs written.

test confusion matrix:


,pred_harmful,pred_unharmful
true_harmful,4,123
true_unharmful,8,92


## 10. Agregirana analiza grešaka (bez sirovog teksta)

Test greške se koriste ISKLJUČIVO za razumevanje, ne za dodavanje/uklanjanje regex obrazaca —
konfiguracija je već zaključana.

In [13]:
def is_obfuscation(aug_type_series):
    return aug_type_series.str.contains("obfuscation", na=False)

for tag, rf in [("validation", validation_results_full), ("test", test_results_full)]:
    obf = is_obfuscation(rf["augmentation_type"])
    harmful_mask = rf["final_label"] == POSITIVE
    recall_obf = (rf.loc[harmful_mask & obf, "prediction"] == POSITIVE).mean() if (harmful_mask & obf).any() else float("nan")
    recall_non_obf = (rf.loc[harmful_mask & ~obf, "prediction"] == POSITIVE).mean() if (harmful_mask & ~obf).any() else float("nan")
    print(f"[{tag}] harmful recall on obfuscation-augmented rows: {recall_obf:.4f} (n={int((harmful_mask & obf).sum())}) "
          f"vs non-obfuscation rows: {recall_non_obf:.4f} (n={int((harmful_mask & ~obf).sum())})")

print()
for tag, fp, fn in [("validation", val_fp, val_fn), ("test", test_fp, test_fn)]:
    print(f"[{tag}] FP by augmentation_type:", fp["augmentation_type"].value_counts().to_dict())
    print(f"[{tag}] FP by language:", fp["language"].value_counts().to_dict())
    print(f"[{tag}] FN by augmentation_type (top 5):", fn["augmentation_type"].value_counts().head(5).to_dict())
    print(f"[{tag}] FN obfuscation vs non-obfuscation: "
          f"{fn['augmentation_type'].str.contains('obfuscation', na=False).sum()} / {len(fn)}")
    print()

[validation] harmful recall on obfuscation-augmented rows: 0.3833 (n=60) vs non-obfuscation rows: 0.0300 (n=100)
[test] harmful recall on obfuscation-augmented rows: 0.1379 (n=29) vs non-obfuscation rows: 0.0000 (n=98)

[validation] FP by augmentation_type: {'translation': 3, 'original': 3}
[validation] FP by language: {'en': 3, 'hi': 1, 'te': 1, 'it': 1}
[validation] FN by augmentation_type (top 5): {'original': 49, 'translation': 48, 'obfuscation_en': 17, 'obfuscation_tr': 9, 'obfuscation_promptonly_tr': 6}
[validation] FN obfuscation vs non-obfuscation: 37 / 134

[test] FP by augmentation_type: {'translation': 4, 'original': 4}
[test] FP by language: {'en': 4, 'id': 2, 'cs': 1, 'pt': 1}
[test] FN by augmentation_type (top 5): {'original': 50, 'translation': 48, 'obfuscation_promptonly_tr': 8, 'obfuscation_promptonly_en': 6, 'obfuscation_tr': 6}
[test] FN obfuscation vs non-obfuscation: 25 / 123



## 11. Poređenje sa ostalim sistemima (test, end-to-end)

Učitavamo SAČUVANE test predikcije za sva 4 druga sistema (nema ponovnog pokretanja bilo kog
modela), potvrđujemo identičan `row_id`/`final_label` skup, i preračunavamo end-to-end metrike
istom metodologijom (nema hardkodovanja ranijih brojeva).

In [14]:
OTHER_SYSTEMS = {
    "Gemma zero-shot": "/home/mls01/scripts/model/results/gemma_demo_zeroshot_v2_no_refusal/test_results_full.csv",
    "Gemma initial LoRA": "/home/mls01/scripts/model/results/gemma_lora_v2_exp1_r8_lr2e4_seed42_max8_es2/test_results_full.csv",
    "Gemma sweep LoRA": "/home/mls01/scripts/model/results/gemma_lora_v2_multiseed/final_test_seed42/test_results_full.csv",
    "Qwen3Guard": "/home/mls01/scripts/model/results/qwen3guard_native_v2_no_refusal/test_results_full.csv",
}

test_row_ids = set(test_df["row_id"])
test_label_map = dict(zip(test_df["row_id"], test_df["final_label"]))

def compute_metrics_end_to_end(y_true, y_pred, positive=POSITIVE):
    tp = fp = fn = tn = invalid = 0
    for t, p in zip(y_true, y_pred):
        if p == "invalid":
            invalid += 1
            if t == positive:
                fn += 1
            else:
                fp += 1
            continue
        if t == positive and p == positive:
            tp += 1
        elif t != positive and p == positive:
            fp += 1
        elif t == positive and p != positive:
            fn += 1
        else:
            tn += 1
    n = len(y_true)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    fnr = fn / (fn + tp) if (fn + tp) else 0.0
    accuracy = (tp + tn) / n if n else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "accuracy": accuracy, "fpr": fpr, "fnr": fnr,
            "invalid_count": invalid, "invalid_rate": invalid / n if n else 0.0}

comparison_rows = [{"System": "Regex baseline", **compute_metrics_end_to_end(
    test_results_full["final_label"].tolist(), test_results_full["prediction"].tolist())}]

for name, path in OTHER_SYSTEMS.items():
    other_df = pd.read_csv(path)
    assert set(other_df["row_id"]) == test_row_ids, f"{name}: row_id set mismatch with test.jsonl"
    assert all(test_label_map[r] == l for r, l in zip(other_df["row_id"], other_df["final_label"])), \
        f"{name}: final_label mismatch with test.jsonl"
    comparison_rows.append({"System": name, **compute_metrics_end_to_end(
        other_df["final_label"].tolist(), other_df["prediction"].tolist())})

comparison_df = pd.DataFrame(comparison_rows).rename(columns={"recall": "harmful_recall"})
comparison_df.to_csv(OUT_DIR / "comparison_all_models_test.csv", index=False)
print("[OK] svih 5 sistema koristi identičnih 227 row_id + final_label vrednosti (test.jsonl).")
display_cols = ["System", "precision", "harmful_recall", "f1", "fpr", "fnr", "accuracy", "invalid_rate"]
comparison_df[display_cols]

[OK] svih 5 sistema koristi identičnih 227 row_id + final_label vrednosti (test.jsonl).


,System,precision,harmful_recall,f1,fpr,fnr,accuracy,invalid_rate
0,Regex baseline,0.333333,0.031496,0.057554,0.08,0.968504,0.422907,0.000000
1,Gemma zero-shot,0.725000,0.913386,0.808362,0.44,0.086614,0.757709,0.022026
2,Gemma initial LoRA,0.888889,0.944882,0.916031,0.15,0.055118,0.903084,0.000000
3,Gemma sweep LoRA,0.927419,0.905512,0.916335,0.09,0.094488,0.907489,0.000000
4,Qwen3Guard,0.945736,0.960630,0.953125,0.07,0.039370,0.947137,0.000000


## 12. REPORT.md

In [15]:
locked_patterns_df["_frag_len"] = locked_patterns_df["expression"].str.replace(" ", "", regex=False).str.len()
n_short_locked = int((locked_patterns_df["_frag_len"] <= 2).sum())
n_long_locked = int((locked_patterns_df["_frag_len"] >= 4).sum())
ngram_size_counts_locked = locked_patterns_df["ngram_size"].value_counts().to_dict()
real_word_ranks = candidate_df.loc[candidate_df["expression"].str.replace(" ", "", regex=False).str.len() >= 4, "rank"]
first_real_word_rank = int(real_word_ranks.min()) if len(real_word_ranks) else None
some_example_real_words = candidate_df[candidate_df["rank"].between(first_real_word_rank, first_real_word_rank + 400)]
some_example_real_words = some_example_real_words[
    some_example_real_words["expression"].str.replace(" ", "", regex=False).str.len() >= 4
].head(10)["expression"].tolist()

val_obf_recall = (validation_results_full.loc[
    (validation_results_full["final_label"] == POSITIVE) & is_obfuscation(validation_results_full["augmentation_type"]),
    "prediction"] == POSITIVE).mean()
val_nonobf_recall = (validation_results_full.loc[
    (validation_results_full["final_label"] == POSITIVE) & ~is_obfuscation(validation_results_full["augmentation_type"]),
    "prediction"] == POSITIVE).mean()
test_obf_recall = (test_results_full.loc[
    (test_results_full["final_label"] == POSITIVE) & is_obfuscation(test_results_full["augmentation_type"]),
    "prediction"] == POSITIVE).mean()
test_nonobf_recall = (test_results_full.loc[
    (test_results_full["final_label"] == POSITIVE) & ~is_obfuscation(test_results_full["augmentation_type"]),
    "prediction"] == POSITIVE).mean()

report_lines = []
report_lines.append("# Train-derived keyword/regex baseline\n")
report_lines.append(
    "Ovo je **train-derived keyword/regex baseline** — regex obrasci nisu ručno napisani niti "
    "preuzeti iz spoljne liste harmful reči, i ovo nije neuronski/ML model. Svi izrazi su "
    "automatski izvedeni isključivo iz `train.jsonl` statistikom (group-aware document frequency "
    "+ smoothed log-odds), po unapred zaključanim pravilima navedenim ispod.\n"
)
report_lines.append("## Cilj\n")
report_lines.append(
    "Jednostavan, reproduktibilan baseline za v2 (bez refusal-a) harmful/unharmful klasifikaciju, "
    "kao donja granica za poređenje sa Gemma zero-shot/LoRA i Qwen3Guard sistemima. Regex provera "
    "i `prompt` i `response`; pozitivna klasa je `harmful`.\n"
)
report_lines.append("## Zašto izraze učimo samo iz train skupa\n")
report_lines.append(
    "Validation i test moraju ostati nezagađeni informacijom korišćenom za konstrukciju modela — "
    "izvlačenje kandidata isključivo iz train skupa je analogno tome da se model trenira samo na "
    "train podacima. Validation se koristi SAMO za izbor top_k/min_distinct_matches konfiguracije "
    "(ne za generisanje izraza), a test se učitava tek nakon što je konfiguracija potpuno "
    "zaključana.\n"
)
report_lines.append("## Normalizacija (zaključana)\n")
report_lines.append(
    "1. Unicode NFKC\n2. lowercase\n3. zamena interpunkcije/ostalih non-word separatora jednim "
    "razmakom (regex [^\\w]+ -> ' ')\n4. spajanje višestrukih razmaka\n"
    "5. uklanjanje početnih/završnih razmaka\n\n"
    "Bez prevođenja, stemming/lemmatizacije, LLM-a, semantic embeddings-a, ručnog dekodiranja "
    "obfuskacija ili spoljnih rečnika.\n"
)
report_lines.append("## Group-aware izvlačenje kandidata\n")
report_lines.append(
    f"Iz normalizovanog prompt+response svakog train reda izvučeni su word n-grami dužine "
    f"{list(NGRAM_SIZES)}. Unija n-grama po redu (prompt + response, svaki izraz doprinosi najviše "
    f"jednom po redu), zatim unija po original_idx grupi (svaki izraz doprinosi najviše jednom po "
    f"grupi, bez obzira na broj redova/augmentacija u grupi ili broj ponavljanja unutar reda). "
    f"Train: {N_HARMFUL_GROUPS} harmful grupa, {N_UNHARMFUL_GROUPS} unharmful grupa. "
    f"Filteri kandidata: harmful_group_df >= {MIN_HARMFUL_GROUP_DF}, "
    f"harmful_group_prevalence > unharmful_group_prevalence, izraz nije potpuno numerički. "
    f"Rezultat: **{len(candidate_df)} kandidata** iz {len(all_expr)} ukupno distinct n-grama u "
    f"train grupama.\n"
)
report_lines.append("## Formula rangiranja\n")
report_lines.append(
    f"Fiksno Jeffreys-stil (add-{ALPHA}) smoothing na group-presence brojevima:\n\n"
    f"```\nsmoothed_log_odds = ln((a+{ALPHA})/(H-a+{ALPHA})) - ln((b+{ALPHA})/(U-b+{ALPHA}))\n"
    f"a = harmful_group_df, b = unharmful_group_df\n"
    f"H = {N_HARMFUL_GROUPS} (n_harmful_train_groups), U = {N_UNHARMFUL_GROUPS} (n_unharmful_train_groups)\n"
    f"```\n\nKandidati su rangirani opadajuće po smoothed_log_odds (tie-break: veći "
    f"harmful_group_df, zatim alfabetski po izrazu, radi potpune determinističnosti). Rangiranje "
    f"se NE menja ručno nakon sortiranja.\n"
)
report_lines.append("## Regex konstrukcija\n")
report_lines.append(
    "Svaki izraz je re.escape-ovan i uokviren (?<!\\w)...(?!\\w) granicama, tako da kratak izraz "
    "ne može pogoditi deo duže reči (npr. izraz 'cat' ne pogađa 'category').\n"
)
report_lines.append("## Validation grid i rezultati\n")
report_lines.append(
    f"Mreža: top_k_patterns={TOP_K_GRID} x min_distinct_matches={MIN_DISTINCT_MATCHES_GRID} = "
    f"{len(TOP_K_GRID)*len(MIN_DISTINCT_MATCHES_GRID)} konfiguracija (nijedna preskočena — "
    f"{len(candidate_df)} kandidata je više od svake testirane top_k vrednosti).\n\n```\n"
    + validation_search_df[["top_k_patterns","min_distinct_matches","precision","recall","f1","fpr",
                            "n_patterns_used"]].to_string(index=False)
    + "\n```\n"
)
report_lines.append("## Zaključana konfiguracija\n")
report_lines.append(
    f"Izabrano prema redosledu (max F1 -> max recall -> min FPR -> min broj obrazaca -> veći "
    f"min_distinct_matches na potpunom tie-u): **top_k_patterns={SELECTED_TOP_K}, "
    f"min_distinct_matches={SELECTED_MIN_DM}** (validation F1={selected['f1']:.4f}).\n\n"
    f"SHA256 zaključane liste od {len(locked_patterns_df)} obrazaca: `{locked_hash}` — potvrđeno "
    f"nepromenjen pre i posle test evaluacije.\n"
)
report_lines.append("## Validation i test metrike (zaključana konfiguracija)\n")
report_lines.append(
    "Regex baseline nema invalid izlaz (uvek vraća harmful ili unharmful), pa su **valid-only i "
    "end-to-end metrike identične** (invalid_count=0, invalid_rate=0).\n\n"
    f"**Validation** ({len(validation_results_full)} redova): precision={val_m['precision']:.4f}, "
    f"recall={val_m['recall']:.4f}, F1={val_m['f1']:.4f}, FPR={val_m['fpr']:.4f}, "
    f"FNR={val_m['fnr']:.4f}, accuracy={val_m['accuracy']:.4f}, MCC={val_m['mcc']:.4f}, "
    f"TP={val_m['tp']} FP={val_m['fp']} FN={val_m['fn']} TN={val_m['tn']}.\n\n"
    f"**Test** ({len(test_results_full)} redova, evaluiran TAČNO JEDNOM posle zaključavanja): "
    f"precision={test_m['precision']:.4f}, recall={test_m['recall']:.4f}, F1={test_m['f1']:.4f}, "
    f"FPR={test_m['fpr']:.4f}, FNR={test_m['fnr']:.4f}, accuracy={test_m['accuracy']:.4f}, "
    f"MCC={test_m['mcc']:.4f}, TP={test_m['tp']} FP={test_m['fp']} FN={test_m['fn']} TN={test_m['tn']}.\n\n"
    f"Test confusion matrix:\n```\n{test_cm.to_string()}\n```\n\n"
    f"**Značajan pad validation->test F1 ({val_m['f1']:.4f} -> {test_m['f1']:.4f})** — objašnjenje "
    f"ispod u sekciji Ograničenja.\n"
)
report_lines.append("## Broj obrazaca i najčešće vrste pogodaka (agregirano)\n")
report_lines.append(
    f"Zaključana lista ima {len(locked_patterns_df)} obrazaca, raspodela po ngram_size: "
    f"{ngram_size_counts_locked}.\n\n"
    f"**Ključan nalaz**: {n_short_locked}/{len(locked_patterns_df)} "
    f"({n_short_locked/len(locked_patterns_df)*100:.0f}%) zaključanih obrazaca su fragmenti dužine "
    f"<=2 karaktera (bez razmaka) — npr. dvoslovni parovi kao 'a t', 'a n', 'i n'. **0 od "
    f"{len(locked_patterns_df)}** zaključanih obrazaca su 'realne reči' od >=4 karaktera. Prvi "
    f"kandidat sa >=4 karaktera (bez razmaka) u celokupnoj rangiranoj listi je na rangu "
    f"**{first_real_word_rank}** — dakle NIJE dostignut ni jednom testiranom top_k vrednošću "
    f"(maks. testiran top_k={max(TOP_K_GRID)}). Primeri takvih 'pravih' kandidata (rang>="
    f"{first_real_word_rank}, ilustrativno, NISU u zaključanoj listi): {some_example_real_words}.\n"
)
report_lines.append("## Uzrok: obfuskacija je u ovom datasetu isključivo harmful\n")
report_lines.append(
    "Programski potvrđeno: svi redovi sa augmentation_type koji sadrži 'obfuscation' imaju "
    "final_label='harmful' u sva tri splita (train/validation/test) — obfuskacija je u v2 "
    "datasetu isključivo primenjena na harmful promptove (verovatno kao adversarial augmentacija "
    "za testiranje jailbreak-bypass-a). Neki obfuskacioni stilovi (npr. wide_spacing, vaporwave) "
    "ubacuju bukvalne razmake između svakog slova originalnog teksta. Pod propisanom jednostavnom "
    "whitespace-tokenizacijom, to razbija reči na pojedinačna slova, pa se karakter-bigrami/"
    "trigrami tih slova (npr. 'a t') pojavljuju u desetinama harmful train grupa i skoro nikad u "
    "unharmful grupama — ne zato što nose harmful semantiku, već zato što se ta specifična "
    "obfuskacija dešava samo na harmful sadržaju. Ovo NIJE bug u kodu — ovo je stvarno svojstvo "
    "dataset-a u kombinaciji sa propisanom jednostavnom normalizacijom (koja ne sme ručno "
    "dekodirati obfuskacije).\n"
)
report_lines.append("## FP/FN analiza (agregirano)\n")
report_lines.append(
    f"Recall na obfuskovanim harmful redovima naspram ne-obfuskovanih (razlika potvrđuje uzrok "
    f"iznad):\n\n"
    f"- validation: {val_obf_recall*100:.1f}% (obfuskacija) vs {val_nonobf_recall*100:.1f}% (bez obfuskacije)\n"
    f"- test: {test_obf_recall*100:.1f}% (obfuskacija) vs {test_nonobf_recall*100:.1f}% (bez "
    f"obfuskacije) — baseline praktično ne detektuje harmful sadržaj koji nije "
    f"karakter-spacing-obfuskovan na test skupu.\n\n"
    f"FN (validation, {len(val_fn)} redova) po augmentation_type (top 5): "
    f"{val_fn['augmentation_type'].value_counts().head(5).to_dict()}\n\n"
    f"FN (test, {len(test_fn)} redova) po augmentation_type (top 5): "
    f"{test_fn['augmentation_type'].value_counts().head(5).to_dict()}\n\n"
    f"FP (validation, {len(val_fp)} redova) po augmentation_type: "
    f"{val_fp['augmentation_type'].value_counts().to_dict()}, po jeziku: "
    f"{val_fp['language'].value_counts().to_dict()}\n\n"
    f"FP (test, {len(test_fp)} redova) po augmentation_type: "
    f"{test_fp['augmentation_type'].value_counts().to_dict()}, po jeziku: "
    f"{test_fp['language'].value_counts().to_dict()}\n\n"
    "Test greške nisu korišćene za dodavanje/uklanjanje regex obrazaca — konfiguracija je ostala "
    "zaključana pre i posle testa (potvrđeno SHA256 hash-om).\n"
)
report_lines.append("## Poređenje sa ostalim sistemima (test, end-to-end)\n")
report_lines.append(
    "Pre poređenja programski potvrđeno: svih 5 sistema koristi identičnih 227 row_id vrednosti i "
    "istu final_label kolonu iz istog v2 test skupa. Metrike za sva 4 postojeća sistema su "
    "PRERAČUNATE iz njihovih sačuvanih test_results_full.csv fajlova istom "
    "compute_metrics_end_to_end funkcijom (nisu hardkodovane), bez ponovnog pokretanja bilo kog "
    "modela.\n\n```\n" + comparison_df[display_cols].to_string(index=False) + "\n```\n"
)
report_lines.append("## Ograničenja keyword/regex pristupa\n")
report_lines.append(
    "- **Nema semantičkog razumevanja** — baseline ne razlikuje harmful i benign upotrebu istih "
    "reči/fraza, i ne generalizuje na parafraze.\n"
    "- **Group-level statistika na ovom datasetu je dominirana artefaktom formatiranja, ne "
    "sadržajem**: pošto je obfuskacija u v2 datasetu isključivo harmful, a propisana jednostavna "
    "tokenizacija ne razlikuje stvarni razmak između reči od razmaka ubačenog unutar obfuskovane "
    f"reči, ceo zaključani top-{SELECTED_TOP_K} sastoji se od kratkih karakter-fragmenata, a ne od "
    f"semantičkih harmful termina (prvi takav termin je na rangu {first_real_word_rank}, van "
    "dostignutog opsega).\n"
    f"- **Slaba generalizacija train->validation->test**: F1 pada sa {selected['f1']:.4f} "
    f"(validation) na {test_m['f1']:.4f} (test) jer se koja konkretna obfuskaciona pod-vrsta (npr. "
    "wide_spacing/vaporwave sa bukvalnim razmacima) nalazi u kom splitu razlikuje po slučaju "
    "uzorkovanja grupa, a ne po stabilnom semantičkom signalu.\n"
    f"- **Praktično nikakva detekcija ne-obfuskovanog harmful sadržaja na testu** "
    f"({test_nonobf_recall*100:.1f}% recall) — baseline je u praksi gotovo isključivo detektor "
    "'da li tekst sadrži karakter-po-karakter razmaknutu obfuskaciju', ne detektor harmful "
    "sadržaja.\n"
    "- **Nema obrade konteksta** (npr. 'the $x example' iz CLAUDE.md) — regex gleda prompt i "
    "response nezavisno, red po red, bez ikakvog multi-turn razumevanja.\n"
    "- **Osetljivo na normalizaciju** — drugačija (agresivnija ili obfuskacija-svesna) "
    "normalizacija bi promenila kandidate i rezultate, ali bi izašla iz okvira 'jednostavnog' "
    "baseline-a specificiranog za ovaj eksperiment.\n"
)
report_lines.append("## Potvrda metodologije\n")
report_lines.append(
    "Test skup NIJE korišćen za generisanje izraza (izrazi su izvedeni isključivo iz train.jsonl) "
    "niti za izbor top_k/min_distinct_matches (izabrano isključivo na validation skupu). Test je "
    "učitan i evaluiran TAČNO JEDNOM, nakon zaključavanja locked_regex_config.json/"
    "locked_patterns.csv, i SHA256 zaključane liste ostaje identičan pre i posle test evaluacije. "
    "Nijedan GPU model nije pokretan; ovaj notebook je čisto CPU (pandas/re/math).\n"
)

(OUT_DIR / "REPORT.md").write_text("\n".join(report_lines))
print(f"[OK] REPORT.md written ({(OUT_DIR / 'REPORT.md').stat().st_size} bytes) -> {OUT_DIR / 'REPORT.md'}")

[OK] REPORT.md written (11084 bytes) -> /home/mls01/scripts/model/results/regex_baseline_v2_no_refusal/REPORT.md


## 13. Završni rezime (terminal)

In [16]:
print("=" * 90)
print("FINAL SUMMARY")
print("=" * 90)
print(f"1. Broj train kandidata (posle filtera): {len(candidate_df)}")
print(f"2. Zaključan broj regex obrazaca: {SELECTED_TOP_K}, threshold (min_distinct_matches): {SELECTED_MIN_DM}")
print(f"3. Validation metrike (zaključana konfiguracija): "
      f"P={val_m['precision']:.4f} R={val_m['recall']:.4f} F1={val_m['f1']:.4f} FPR={val_m['fpr']:.4f} "
      f"invalid_rate={val_m['invalid_rate']:.4f}")
print(f"4. Test metrike (jednokratna evaluacija): "
      f"P={test_m['precision']:.4f} R={test_m['recall']:.4f} F1={test_m['f1']:.4f} FPR={test_m['fpr']:.4f} "
      f"invalid_rate={test_m['invalid_rate']:.4f}")
print(f"5. Test confusion matrix:\n{test_cm.to_string()}")
print(f"\n6. Poređenje svih pet sistema (test, end-to-end):")
print(comparison_df[display_cols].to_string(index=False))
print(f"\n7. Hash zaključane regex liste nepromenjen pre/posle testa: {hash_after_test == locked_hash} "
      f"({locked_hash})")
print(f"8. Putanje:")
print(f"   notebook: scripts/model/regex_baseline.ipynb")
print(f"   results:  {OUT_DIR}")
print(f"\ntotal runtime: {time.time()-t0:.1f}s")

FINAL SUMMARY
1. Broj train kandidata (posle filtera): 16139
2. Zaključan broj regex obrazaca: 400, threshold (min_distinct_matches): 1
3. Validation metrike (zaključana konfiguracija): P=0.8125 R=0.1625 F1=0.2708 FPR=0.0606 invalid_rate=0.0000
4. Test metrike (jednokratna evaluacija): P=0.3333 R=0.0315 F1=0.0576 FPR=0.0800 invalid_rate=0.0000
5. Test confusion matrix:
                pred_harmful  pred_unharmful
true_harmful               4             123
true_unharmful             8              92

6. Poređenje svih pet sistema (test, end-to-end):
            System  precision  harmful_recall       f1  fpr      fnr  accuracy  invalid_rate
    Regex baseline   0.333333        0.031496 0.057554 0.08 0.968504  0.422907      0.000000
   Gemma zero-shot   0.725000        0.913386 0.808362 0.44 0.086614  0.757709      0.022026
Gemma initial LoRA   0.888889        0.944882 0.916031 0.15 0.055118  0.903084      0.000000
  Gemma sweep LoRA   0.927419        0.905512 0.916335 0.09 0.094488  